# wilro-sync on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wiltechs-hd/wilro-sync/blob/main/notebooks/wilro_sync_colab.ipynb)

Plain-code walkthrough for [wilro-sync](https://github.com/wiltechs-hd/wilro-sync): set up → sanity checks →
prepare clips → train → lip-sync. Edit the configuration cell, then run the cells in order. Everything uses the
Python API, so you can change any setting in code (e.g. `cfg.lr_new = 5e-5`) before running a step.

| Runtime | What works |
|---|---|
| A100 40/80 GB | everything |
| L4 24 GB | training (LoRA, 512×512×49, 8-bit Adam), data prep, inference |
| T4 16 GB | tests, data prep, inference (fp16); not training |

Model downloads (~30 GB) go to local disk; clips, checkpoints and outputs go to `WORK_DIR` on Google Drive.
Which datasets to use: [docs/DATASETS.md](https://github.com/wiltechs-hd/wilro-sync/blob/main/docs/DATASETS.md).

## 1. Configuration

In [ ]:
# Where the code comes from: "github" (default), "drive" (your own copy) or "local" (already at REPO)
CODE_SOURCE = "github"
GITHUB_URL = "https://github.com/wiltechs-hd/wilro-sync.git"
GIT_REF = "main"  # branch, tag or commit; pin a commit to reproduce a run
DRIVE_CODE_DIR = "/content/drive/MyDrive/wilro-sync"
REPO = "/content/wilro-sync"

# Persistent working folder (raw videos, prepared clips, checkpoints, outputs)
WORK_DIR = "/content/drive/MyDrive/wilro-sync-work"

# Data preparation (scripts/prepare_clips.py)
CLIP_SIZE = 512  # square crop, pixels
CLIP_FRAMES = 49  # 1 + 4k
CLIPS_PER_VIDEO = 4
CAPTION = "A person speaking clearly with natural, well-articulated lip and teeth movements."

# SyncNet filtering before preparation (drops badly synced videos, records each video's A/V offset)
SYNC_MIN_CONF = 3.0  # minimum SyncNet confidence
SYNC_MAX_OFFSET = 3  # max |A/V offset| in frames (25 fps); smaller offsets are corrected
SYNC_ASSUME_CROPPED = False  # True if the videos are already tight face crops (e.g. HDTF 512x512): faster

# Training: any TrainConfig field (see wilrosync/train/config.py), applied on top of configs/train/stage_a.yaml
RUN_NAME = "stage_a_colab"
TRAIN_OVERRIDES = {
    "train_mode": "lora",  # lora | adapter | full
    "lora_rank": 64,
    "max_steps": 2000,
    "grad_accum": 8,
    "save_every": 250,
    "keep_last": 2,
}
RESUME = True

# Inference defaults (any InferenceConfig field, see configs/infer/default.yaml)
INFER_OVERRIDES = {"steps": 50, "tau": 0.92, "max_side": 832, "seed": 0}
DSCFG_OVERRIDES = {"omega_peak": 4.5, "gamma": 1.5}

## 2. Environment

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys

os.environ.setdefault("HF_HOME", "/content/hf_cache")  # large model files stay on local disk
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

if "google.colab" in sys.modules and (CODE_SOURCE == "drive" or WORK_DIR.startswith("/content/drive")):
    from google.colab import drive

    drive.mount("/content/drive")
try:  # optional: a Colab secret named HF_TOKEN gives faster, rate-limit-free downloads
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

RAW = os.path.join(WORK_DIR, "raw")
LATENTS = os.path.join(WORK_DIR, "latents")
RUN_DIR = os.path.join(WORK_DIR, "runs", RUN_NAME)
for d in (os.path.join(RAW, "wild"), LATENTS, RUN_DIR, os.path.join(WORK_DIR, "outputs")):
    os.makedirs(d, exist_ok=True)


def sh(cmd: str, check: bool = True) -> int:
    """Run a shell command, streaming its output."""
    print(f"$ {cmd}")
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() and check:
        raise RuntimeError(f"failed ({p.returncode}): {cmd}")
    return p.returncode


def latest_checkpoint(run_dir: str):
    steps = [d for d in sorted(glob.glob(os.path.join(run_dir, "step_*")))
             if os.path.isfile(os.path.join(d, "step.json"))]
    return steps[-1] if steps else None


import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    GPU, GPU_MEM, BF16 = props.name, props.total_memory / 2**30, torch.cuda.is_bf16_supported()
else:
    GPU, GPU_MEM, BF16 = "none", 0.0, False
CAN_TRAIN = BF16 and GPU_MEM >= 22
print(f"GPU {GPU} | {GPU_MEM:.0f} GB | bf16 {BF16} | can train: {CAN_TRAIN}")

## 3. Code and dependencies

In [ ]:
if CODE_SOURCE == "github":
    if os.path.isdir(os.path.join(REPO, ".git")):  # re-run: update in place
        sh(f"git -C {REPO} fetch -q --depth 1 origin {GIT_REF} && git -C {REPO} checkout -q FETCH_HEAD")
    else:
        shutil.rmtree(REPO, ignore_errors=True)
        sh(f"git clone -q {GITHUB_URL} {REPO} && git -C {REPO} checkout -q {GIT_REF}")
    sh(f"git -C {REPO} log -1 --format='code: %h %s (%cd)' --date=short")
elif CODE_SOURCE == "drive":
    shutil.rmtree(REPO, ignore_errors=True)
    shutil.copytree(DRIVE_CODE_DIR, REPO, ignore=shutil.ignore_patterns("__pycache__", "*.egg-info", "runs", "data"))
assert os.path.isfile(os.path.join(REPO, "pyproject.toml")), f"no repo at {REPO}"

os.chdir(REPO)
sys.path.insert(0, REPO)  # editable installs are only seen by new processes
sh(f'pip install -q -e "{".[dev,train]" if torch.cuda.is_available() else ".[dev]"}"')
import wilrosync

print("wilro-sync", wilrosync.__version__)

## 4. Sanity checks

Unit tests (CPU, ~20 s). The real-weights check downloads Wan2.1-1.3B (~6 GB) and confirms the lip-sync wrapper
reproduces it exactly before any training.

In [ ]:
sh("python -m pytest -q")

In [ ]:
sh("python scripts/check_real_wan.py")

## 5. Prepare training clips

Put talking-head videos under `WORK_DIR/raw/wild/` (see [docs/DATASETS.md](https://github.com/wiltechs-hd/wilro-sync/blob/main/docs/DATASETS.md)).
Optional MEAD pseudo pairs: `WORK_DIR/raw/pairs.jsonl` with lines
`{"kind": "pseudo_pair", "cond": "mead/M003/front/neutral/level_1/001.mp4", "target": "mead/M003/front/neutral/level_1/002.mp4"}`.

Clips are centre-cropped, so use face-centred videos (HDTF, MEAD frontal, or pre-cropped TalkVid clips).

Steps: build a manifest → **SyncNet filter** (scores every video, keeps those with confidence ≥ `SYNC_MIN_CONF`
and |offset| ≤ `SYNC_MAX_OFFSET`, and stores each offset so the audio is re-aligned) → prepare clips. SyncNet
scores are saved in a `.report.jsonl` next to the manifest, so re-running skips videos already scored.
The first preparation run downloads the encoders (~28 GB); prepared clips are reused by every training run.

In [ ]:
import importlib.util

VIDEO_EXT = (".mp4", ".mov", ".mkv", ".webm", ".avi")


def build_manifest(folder: str, out_path: str, caption: str = CAPTION) -> int:
    videos = sorted(p for p in glob.glob(os.path.join(folder, "**", "*"), recursive=True)
                    if p.lower().endswith(VIDEO_EXT))
    with open(out_path, "w") as f:
        for p in videos:
            f.write(json.dumps({"kind": "arbitrary", "video": os.path.relpath(p, RAW), "caption": caption}) + "\n")
    return len(videos)


def prepare(manifest: str, out_dir: str, rebuild: bool = False) -> int:
    """Run scripts/prepare_clips.py in-process; skips if already prepared."""
    index = os.path.join(out_dir, "index.jsonl")
    if os.path.isfile(index) and not rebuild:
        n = sum(1 for _ in open(index))
        print(f"{out_dir}: {n} clips already prepared (rebuild=True to redo)")
        return n
    shutil.rmtree(out_dir, ignore_errors=True)
    spec = importlib.util.spec_from_file_location("prepare_clips", os.path.join(REPO, "scripts/prepare_clips.py"))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    n = mod.main(["--manifest", manifest, "--root", RAW, "--out", out_dir,
                  "--size", str(CLIP_SIZE), str(CLIP_SIZE), "--frames", str(CLIP_FRAMES),
                  "--clips-per-video", str(CLIPS_PER_VIDEO), "--device", "cuda" if GPU_MEM else "cpu"])
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return n

In [ ]:
from wilrosync.data.sync_filter import filter_manifest

wild_manifest = os.path.join(WORK_DIR, "wild.jsonl")
print("videos:", build_manifest(os.path.join(RAW, "wild"), wild_manifest))
wild_synced = os.path.join(WORK_DIR, "wild.synced.jsonl")
filter_manifest(wild_manifest, RAW, wild_synced, min_conf=SYNC_MIN_CONF, max_offset=SYNC_MAX_OFFSET,
                assume_cropped=SYNC_ASSUME_CROPPED)

In [ ]:
# distribution of SyncNet scores (all videos, including rejected ones)
import matplotlib.pyplot as plt

report = [json.loads(line) for line in open(wild_synced.replace(".jsonl", ".report.jsonl")) if line.strip()]
scored = [r for r in report if "sync_conf" in r]
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist([r["sync_conf"] for r in scored], bins=30)
ax[0].axvline(SYNC_MIN_CONF, color="r")
ax[0].set_title("SyncNet confidence")
ax[1].hist([r["av_offset"] for r in scored], bins=range(-15, 17))
ax[1].set_title("A/V offset (frames)")
plt.show()

In [ ]:
prepare(wild_synced, os.path.join(LATENTS, "wild"))

pairs_manifest = os.path.join(RAW, "pairs.jsonl")
if os.path.isfile(pairs_manifest):
    pairs_synced = os.path.join(WORK_DIR, "pairs.synced.jsonl")
    filter_manifest(pairs_manifest, RAW, pairs_synced, min_conf=SYNC_MIN_CONF, max_offset=SYNC_MAX_OFFSET,
                    assume_cropped=SYNC_ASSUME_CROPPED)
    prepare(pairs_synced, os.path.join(LATENTS, "pairs"))

## 6. Train

Builds a `TrainConfig` from `configs/train/stage_a.yaml` + `TRAIN_OVERRIDES`; inspect or change `cfg` before calling
`train(cfg)`. Checkpoints go to `RUN_DIR/step_*`; with `RESUME` the latest one is continued after a disconnect.

In [ ]:
from wilrosync.train.config import load_config
from wilrosync.train.trainer import train

pairs_index = os.path.join(LATENTS, "pairs", "index.jsonl")
has_pairs = os.path.isfile(pairs_index)
overrides = {
    "output_dir": RUN_DIR,
    "data.arbitrary_index": os.path.join(LATENTS, "wild", "index.jsonl"),
    "data.pairs_index": pairs_index if has_pairs else "null",
    "data.tds_enabled": has_pairs,  # timestep-dependent sampling needs pseudo pairs
    "data.null_text": os.path.join(LATENTS, "null_text.safetensors"),
    "data.num_workers": 2,
    "mixed_precision": "bf16" if BF16 else "no",
    "optimizer": "adamw" if GPU_MEM >= 60 else "adamw8bit",
    **TRAIN_OVERRIDES,
}
ckpt = latest_checkpoint(RUN_DIR) if RESUME else None
if ckpt:
    overrides["resume_from"] = ckpt
cfg = load_config("configs/train/stage_a.yaml", [f"{k}={v}" for k, v in overrides.items()])
print(f"mode={cfg.train_mode} steps={cfg.max_steps} batch={cfg.batch_size}x{cfg.grad_accum} "
      f"lr={cfg.lr}/{cfg.lr_new} precision={cfg.mixed_precision} optim={cfg.optimizer} resume={cfg.resume_from}")

In [ ]:
assert CAN_TRAIN, f"{GPU} can't train this model (needs bf16 and >= 22 GB): use an L4 or A100 runtime"
result = train(cfg)
print("finished at step", result["steps"])

In [ ]:
import matplotlib.pyplot as plt

log = [json.loads(line) for line in open(os.path.join(RUN_DIR, "log.jsonl")) if line.strip()]
plt.figure(figsize=(8, 3.5))
plt.plot([r["step"] for r in log], [r["loss"] for r in log])
plt.xlabel("step")
plt.ylabel("flow-matching loss")
plt.grid(alpha=0.3)
plt.title(RUN_NAME)
plt.show()

## 7. Lip sync

`lipsync(video, audio, bbox=None)` runs the pipeline with the latest checkpoint of `RUN_DIR` (EMA weights if
present). Pass `bbox=(x0, y0, x1, y1)` in source-video pixels to sync only that face in a multi-face scene;
`preview(video, bbox)` shows the first frame with a grid and the box. Until a checkpoint is properly trained, the
output is not lip-synced.

In [ ]:
from base64 import b64encode

import matplotlib.patches as patches
from IPython.display import HTML, display

from wilrosync.io.media import read_audio, read_video, write_video
from wilrosync.pipeline.lipsync import LipSyncPipeline, load_infer_config
from wilrosync.pipeline.target import TargetSignals

_pipe = {}


def load_pipeline(ckpt: str | None = None, use_ema: bool = True) -> LipSyncPipeline:
    ckpt = ckpt or latest_checkpoint(RUN_DIR)
    assert ckpt, f"no checkpoint in {RUN_DIR}"
    if use_ema and os.path.isdir(os.path.join(ckpt, "ema")):
        ckpt = os.path.join(ckpt, "ema")
    if _pipe.get("ckpt") != ckpt:
        _pipe.clear()
        dtype = torch.bfloat16 if BF16 else torch.float16
        _pipe.update(ckpt=ckpt, pipe=LipSyncPipeline.from_pretrained(ckpt, device="cuda", dtype=dtype))
    return _pipe["pipe"]


def preview(video: str, bbox=None) -> None:
    frame = read_video(video, max_frames=1)[0][0]
    h, w = frame.shape[:2]
    fig, ax = plt.subplots(figsize=(9, 9 * h / w))
    ax.imshow(frame)
    ax.set_xticks(range(0, w, max(1, w // 10)))
    ax.set_yticks(range(0, h, max(1, h // 10)))
    ax.grid(alpha=0.4)
    if bbox:
        x0, y0, x1, y1 = bbox
        ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=2, ec="lime"))
    plt.show()


def lipsync(video: str, audio: str, bbox=None, out: str | None = None, ckpt: str | None = None, **overrides) -> str:
    pipe = load_pipeline(ckpt)
    frames, fps = read_video(video)
    target = TargetSignals.from_bbox(len(frames), *frames.shape[1:3], bbox) if bbox else None
    icfg = load_infer_config("configs/infer/default.yaml")
    for k, v in {**INFER_OVERRIDES, **overrides}.items():
        setattr(icfg, k, v)
    for k, v in DSCFG_OVERRIDES.items():
        setattr(icfg.dscfg, k, v)
    result = pipe(frames, fps, read_audio(audio), target=target, cfg=icfg)
    out = out or os.path.join(WORK_DIR, "outputs", os.path.splitext(os.path.basename(video))[0] + "_synced.mp4")
    write_video(out, result, fps, audio_path=audio)
    print(f"{len(result)} frames -> {out}")
    return out


def show(path: str, width: int = 720) -> None:
    data = b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls src="data:video/mp4;base64,{data}"></video>'))

In [ ]:
# Example: put your files in WORK_DIR/inputs/ and edit the paths / box below.
video = os.path.join(WORK_DIR, "inputs", "two_people.mp4")
audio = os.path.join(WORK_DIR, "inputs", "speech.wav")
bbox = None  # e.g. (410, 80, 620, 330) for the target face; None = whole frame (single speaker)

preview(video, bbox)
out = lipsync(video, audio, bbox=bbox)
show(out)

In [ ]:
# Lip-sync metrics (SyncNet): LSE-C higher is better, LSE-D lower is better; bbox -> target face only
from wilrosync.eval.syncnet import lse

print("source :", lse(video, audio, bbox=bbox))  # original lips vs the new audio (should score badly)
print("output :", lse(out, bbox=bbox))

## Notes

* After a disconnect: run sections 1–3, then section 6 (`RESUME = True`) to continue training.
* Out of memory while training: smaller `CLIP_SIZE` (384) or `CLIP_FRAMES` (33) and re-prepare, or `train_mode="adapter"`.
* Out of memory at inference: `lipsync(..., max_side=640)`.
* Only process footage you have the rights and consent to edit, and label results as synthetic.